# Protein Checkpoint Top-1 Benchmark

Evaluate a protein ProGen checkpoint on held-out `train.txt` lines with causal-LM loss and next-token top-1 accuracy.

In [1]:
import json
import math
import os
import sys
from pathlib import Path
import torch
from tqdm.auto import tqdm

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "libs").is_dir():
            return path
    repo_dir_env = os.environ.get("MDNAC_REPO_DIR")
    if repo_dir_env:
        repo_dir = Path(repo_dir_env).expanduser().resolve()
        if (repo_dir / "pyproject.toml").exists() and (repo_dir / "libs").is_dir():
            return repo_dir
    for candidate in (
        Path("/content/MDNAC"),
        Path("/content/drive/MyDrive/MDNAC"),
    ):
        if (candidate / "pyproject.toml").exists() and (candidate / "libs").is_dir():
            return candidate.resolve()
    raise RuntimeError("Could not locate project root. Set MDNAC_REPO_DIR or clone/mount the repo in Colab.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from libs.notebook_runtime import bootstrap_notebook

RUNTIME = bootstrap_notebook(PROJECT_ROOT)
PROJECT_ROOT = Path(RUNTIME["repo_dir"])

from libs.core import (
    MDCDecoderModel,
    MDCModelConfig,
    compute_mdc_causal_lm_loss,
    create_streaming_protein_lm_dataloader,
    discover_protein_train_text_paths,
    is_supported_protein_checkpoint_family,
    load_protein_pretrain_checkpoint,
)
from libs.data.training.tokenizer import SequenceTokenizer

RUNTIME

c:\Users\Admin\Desktop\MDNAC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'repo_dir': 'C:\\Users\\Admin\\Desktop\\MDNAC',
 'platform': 'Windows',
 'platform_name': 'Windows',
 'is_colab': False,
 'is_notebook': True,
 'python': '3.11.15',
 'cuda_available': True,
 'cuda_device_count': 1}

In [2]:
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "checkpoints" / "protein_from_scratch"
CHECKPOINT_PATH = Path(os.environ.get("MDNAC_EVAL_CHECKPOINT", CHECKPOINT_DIR / "checkpoint_best.pt")).expanduser()
if not CHECKPOINT_PATH.is_absolute():
    CHECKPOINT_PATH = (PROJECT_ROOT / CHECKPOINT_PATH).resolve()
if not CHECKPOINT_PATH.exists() and CHECKPOINT_PATH.name == "checkpoint_best.pt":
    CHECKPOINT_PATH = CHECKPOINT_DIR / "checkpoint_last.pt"

TRAIN_RATIO = 0.9
BATCH_SIZE = 1
MAX_BATCHES = 1000
USE_CUDA_AUTOCAST = True
RESULTS_PATH = PROJECT_ROOT / "data" / "evals" / "protein_top1_benchmark.json"

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Missing checkpoint: {CHECKPOINT_PATH}")

checkpoint_meta = torch.load(CHECKPOINT_PATH, map_location="cpu")
if not is_supported_protein_checkpoint_family(checkpoint_meta.get("model_family")):
    raise ValueError(
        "This benchmark expects a stage-2 protein pretrain checkpoint. "
        f"Received: {checkpoint_meta.get('model_family')!r}"
    )

checkpoint_meta.keys()

dict_keys(['model_family', 'backbone_family', 'model_state_dict', 'optimizer_state_dict', 'model_config', 'tokenizer_map', 'tokenizer_map_path', 'epoch', 'global_step', 'tokens_seen', 'train_losses', 'val_losses', 'training_args', 'best_val_loss', 'best_metric_name', 'corpus_files', 'minio_train_parts_prefix_uri', 'minio_train_part_uris'])

In [3]:
tokenizer_map_path = Path(checkpoint_meta["tokenizer_map_path"])
if not tokenizer_map_path.exists():
    tokenizer_map_path.parent.mkdir(parents=True, exist_ok=True)
    tokenizer_map_path.write_text(
        json.dumps(checkpoint_meta["tokenizer_map"], ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

tokenizer = SequenceTokenizer.load_map(tokenizer_map_path)
train_text_path = Path(
    checkpoint_meta.get("corpus_file")
    or PROJECT_ROOT / "data" / "compiled" / "refseq_bacteria_protein" / "train.txt"
)
primary_parts_dir = PROJECT_ROOT / "data" / "cache" / "protein_train_parts"
primary_train_text_paths = tuple(
    sorted(
        primary_parts_dir.glob("train_part_*.txt"),
        key=lambda path: (0, int(path.stem.rsplit("_", 1)[-1])) if path.stem.rsplit("_", 1)[-1].isdigit() else (1, path.name),
    )
) if primary_parts_dir.exists() else ()
checkpoint_corpus_files = tuple(Path(path) for path in checkpoint_meta.get("corpus_files", ()) if Path(path).exists())
if primary_train_text_paths:
    train_text_paths = primary_train_text_paths
    corpus_source = "primary_parts_dir"
elif checkpoint_corpus_files:
    train_text_paths = checkpoint_corpus_files
    corpus_source = "checkpoint_corpus_files"
else:
    train_text_paths = discover_protein_train_text_paths(
        train_text_path,
        part_glob="train_part_*.txt",
        prefer_parts=True,
    )
    corpus_source = "discovered_from_corpus_file"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_config_payload = dict(checkpoint_meta["model_config"])
if model_config_payload.get("layer_types") is not None:
    model_config_payload["layer_types"] = tuple(model_config_payload["layer_types"])
if device.type != "cuda":
    model_config_payload["dtype"] = torch.float32

model_config = MDCModelConfig(**model_config_payload)
context_length = int(model_config.context_length)
val_loader = create_streaming_protein_lm_dataloader(
    tokenizer,
    context_length=context_length,
    part_paths=train_text_paths,
    stride=max(1, context_length // 2),
    batch_size=BATCH_SIZE,
    split="val",
    train_ratio=TRAIN_RATIO,
    split_seed=42,
    shuffle_parts=False,
    shuffle_examples=False,
    num_workers=0,
    pin_memory=device.type == "cuda",
)

model = MDCDecoderModel(model_config).to(device)
checkpoint = load_protein_pretrain_checkpoint(
    CHECKPOINT_PATH,
    model=model,
    optimizer=None,
    map_location=device,
)

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Corpus: {train_text_path}")
print(f"Primary parts dir: {primary_parts_dir}")
print(f"Corpus source: {corpus_source}")
print(f"Corpus files: {len(train_text_paths)}")
print(f"Tokenizer vocab: {tokenizer.vocab_size}")
print(f"Benchmark source files: {len(train_text_paths)}")
print(f"Streaming validation: split=val train_ratio={TRAIN_RATIO} max_batches={MAX_BATCHES}")
print(f"Batch size: {BATCH_SIZE} device={device} autocast={USE_CUDA_AUTOCAST and device.type == 'cuda'}")

The MDC fast path is unavailable (missing optional libraries: causal-conv1d, flash-linear-attention). Falling back to the torch implementation. The fallback uses fp32 computation (2x VRAM for activations).
Checkpoint: C:\Users\Admin\Desktop\MDNAC\data\checkpoints\protein_from_scratch\checkpoint_best.pt
Corpus: C:\Users\Admin\Desktop\MDNAC\data\compiled\refseq_bacteria_protein\train.txt
Primary parts dir: C:\Users\Admin\Desktop\MDNAC\data\cache\protein_train_parts
Corpus source: primary_parts_dir
Corpus files: 5
Tokenizer vocab: 256
Benchmark source files: 5
Streaming validation: split=val train_ratio=0.9 max_batches=1000
Batch size: 1 device=cuda autocast=True


In [4]:
model.eval()
losses = []
correct = 0
total = 0
autocast_dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
use_autocast = USE_CUDA_AUTOCAST and device.type == "cuda"
progress_bar = tqdm(total=MAX_BATCHES, desc="Top-1 benchmark", unit="batch")

try:
    with torch.no_grad():
        for batch_index, batch in enumerate(val_loader):
            if batch_index >= MAX_BATCHES:
                break

            batch = type(batch)(
                input_ids=batch.input_ids.to(device),
                attention_mask=batch.attention_mask.to(device),
                labels=batch.labels.to(device),
            )
            with torch.amp.autocast("cuda", dtype=autocast_dtype, enabled=use_autocast):
                logits = model(batch.input_ids, attn_mask=batch.attention_mask)
                loss = compute_mdc_causal_lm_loss(logits, batch.labels)
            losses.append(float(loss.item()))

            predictions = logits.argmax(dim=-1)
            valid_mask = batch.labels != -100
            correct += int((predictions[valid_mask] == batch.labels[valid_mask]).sum().item())
            total += int(valid_mask.sum().item())

            running_loss = sum(losses) / len(losses)
            running_top1 = correct / total if total else float("nan")
            progress_bar.update(1)
            progress_bar.set_postfix(loss=f"{running_loss:.4f}", top1=f"{running_top1:.4f}", tokens=total)
finally:
    progress_bar.close()

mean_loss = sum(losses) / len(losses) if losses else float("nan")
results = {
    "checkpoint_path": str(CHECKPOINT_PATH.resolve()),
    "corpus_file": str(train_text_path.resolve()),
    "corpus_source": corpus_source,
    "corpus_files": [str(path.resolve()) for path in train_text_paths],
    "model_family": checkpoint.get("model_family"),
    "backbone_family": checkpoint.get("backbone_family"),
    "batches_evaluated": len(losses),
    "tokens_evaluated": total,
    "streaming_eval": True,
    "batch_size": BATCH_SIZE,
    "autocast": use_autocast,
    "autocast_dtype": str(autocast_dtype) if use_autocast else None,
    "loss": mean_loss,
    "perplexity": math.exp(mean_loss) if math.isfinite(mean_loss) else float("nan"),
    "top1_accuracy": correct / total if total else float("nan"),
}

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.write_text(json.dumps(results, indent=2) + "\n", encoding="utf-8")
results

Top-1 benchmark: 100%|██████████| 1000/1000 [08:24<00:00,  1.98batch/s, loss=3.7844, tokens=177202, top1=0.2378]


{'checkpoint_path': 'C:\\Users\\Admin\\Desktop\\MDNAC\\data\\checkpoints\\protein_from_scratch\\checkpoint_best.pt',
 'corpus_file': 'C:\\Users\\Admin\\Desktop\\MDNAC\\data\\compiled\\refseq_bacteria_protein\\train.txt',
 'corpus_source': 'primary_parts_dir',
 'corpus_files': ['C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\protein_train_parts\\train_part_1.txt',
  'C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\protein_train_parts\\train_part_2.txt',
  'C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\protein_train_parts\\train_part_3.txt',
  'C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\protein_train_parts\\train_part_4.txt',
  'C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\protein_train_parts\\train_part_5.txt'],
 'model_family': 'progen_protein_lm',
 'backbone_family': 'progen',
 'batches_evaluated': 1000,
 'tokens_evaluated': 177202,
 'streaming_eval': True,
 'batch_size': 1,
 'autocast': True,
 'autocast_dtype': 'torch.bfloat16',
 'loss': 3.784400765180588,
 'perplexity': 44.009